# Full-gene LOCO results assembly

Consolidation, independent integrity checks, comparative metrics, and manuscript figures from the four saved Kaggle cohort runs. The notebook does not refit models.

In [1]:
import hashlib
import json
import os
import tempfile
import zipfile
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, confusion_matrix,
    brier_score_loss, matthews_corrcoef, roc_curve, precision_recall_curve
)

if "CMPS355_PROJECT_ROOT" in os.environ:
    PROJECT_ROOT = Path(os.environ["CMPS355_PROJECT_ROOT"]).expanduser().resolve()
else:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/CMPS355_Research")

INPUT = PROJECT_ROOT / "runs"
OUTPUT_ROOT = INPUT / "full_gene_results_report"
if not INPUT.is_dir():
    raise FileNotFoundError(f"Missing project runs directory: {INPUT}")

EXPECTED_DATASET = "8a83f20035fcac5ae5d8bfe496e4d7491e47400b61ca1080b828c8ad54e91a74"
COHORTS = ("d1", "d2", "d3", "d4")
METHODS = ("lasso", "elastic_net")
EXPECTED_N = dict(zip(COHORTS, (260, 179, 76, 174)))
SEED = 355
BOOTSTRAPS = 1000
rng = np.random.default_rng(SEED)

run_dir = OUTPUT_ROOT / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir.mkdir(parents=True, exist_ok=False)

def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def save_csv(df, name):
    df.to_csv(run_dir / name, index=False)

def save_plot(fig, name):
    fig.savefig(run_dir / (name + ".png"), bbox_inches="tight", dpi=300)
    fig.savefig(run_dir / (name + ".pdf"), bbox_inches="tight")
    plt.close(fig)

print("Project:", PROJECT_ROOT)
print("Searching:", INPUT)
print("Output:", run_dir)

Mounted at /content/drive
Project: /content/drive/MyDrive/CMPS355_Research
Searching: /content/drive/MyDrive/CMPS355_Research/runs
Output: /content/drive/MyDrive/CMPS355_Research/runs/full_gene_results_report/20261107T091750523128Z


In [2]:
sources = {}
origins = {}

def add_source(report_path, origin):
    try:
        report = json.loads(report_path.read_text(encoding="utf-8"))
    except (UnicodeError, json.JSONDecodeError):
        return
    fold = report.get("heldout")
    if fold not in COHORTS or "artifacts_sha256" not in report or "config" not in report:
        return
    if report.get("status") not in {"PASS", "WARN_CONVERGENCE"}:
        raise ValueError(f"Unsuccessful run for {fold}: {report.get('status')} at {origin}")
    if report.get("dataset", {}).get("zip_sha256") != EXPECTED_DATASET:
        raise ValueError(f"Dataset checksum mismatch in {fold}: {origin}")
    folder = report_path.parent
    for name in ("predictions.csv", "inner_scores.csv", "selected_genes.csv"):
        path = folder / name
        if not path.is_file() or digest(path) != report["artifacts_sha256"].get(name):
            raise ValueError(f"Missing or checksum-invalid {fold}/{name} in {origin}")
    checksum = digest(report_path)
    if fold in sources:
        if sources[fold][2] != checksum:
            raise ValueError(f"Conflicting {fold} results in {origins[fold]} and {origin}")
        return
    sources[fold] = (report, folder, checksum)
    origins[fold] = origin

for report_path in sorted(INPUT.rglob("report.json")):
    if "full_gene_loco" in report_path.parts:
        add_source(report_path, str(report_path.relative_to(PROJECT_ROOT)))

zip_scratch = Path(tempfile.mkdtemp(prefix="cmps355_loco_inputs_"))
for archive in sorted(INPUT.rglob("*.zip")):
    if archive.stat().st_size > 100_000_000:
        continue
    try:
        with zipfile.ZipFile(archive) as z:
            reports = [n for n in z.namelist() if n.endswith("/report.json") and "full_gene_loco/" in n]
            for report_name in reports:
                prefix = report_name.rsplit("/", 1)[0]
                fold = prefix.split("/")[-1]
                if fold not in COHORTS:
                    continue
                folder = zip_scratch / hashlib.sha256(str(archive).encode()).hexdigest()[:12] / fold
                folder.mkdir(parents=True, exist_ok=True)
                for name in ("report.json", "predictions.csv", "inner_scores.csv", "selected_genes.csv"):
                    entry = prefix + "/" + name
                    if entry in z.namelist():
                        (folder / name).write_bytes(z.read(entry))
                add_source(folder / "report.json", f"{archive.relative_to(PROJECT_ROOT)}::{prefix}")
    except (zipfile.BadZipFile, OSError) as exc:
        print("Skipped unreadable archive:", archive.name, str(exc))

if set(sources) != set(COHORTS):
    raise ValueError(f"Expected D1–D4 results; found {sorted(sources)}. Place the four Kaggle output folders or ZIPs anywhere under {INPUT}.")

reference_config = sources[COHORTS[0]][0]["config"]
reference_files = sources[COHORTS[0]][0]["dataset"].get("csv_sha256")
for fold in COHORTS:
    report = sources[fold][0]
    if report["config"] != reference_config:
        raise ValueError(f"Model configurations differ between cohort runs: {fold}")
    if report["dataset"].get("csv_sha256") != reference_files:
        raise ValueError(f"Underlying input CSV hashes differ for cohort {fold}")
    print(f"{fold.upper()}: verified {origins[fold]}")

all_predictions = []
all_inner = []
all_genes = []
all_parameters = []
software_versions = {}
convergence_warnings = []
for fold in COHORTS:
    report, folder, checksum = sources[fold]
    pred = pd.read_csv(folder / "predictions.csv", dtype={"ID": str, "Study": str, "class_true": str})
    inner = pd.read_csv(folder / "inner_scores.csv")
    genes = pd.read_csv(folder / "selected_genes.csv")
    if not {"outer", "method", "ID", "Study", "class_true", "y_true", "probability_CD"} <= set(pred):
        raise ValueError(f"Missing prediction fields in {fold}")
    if set(pred["outer"]) != {fold} or set(pred["method"]) != set(METHODS):
        raise ValueError(f"Incorrect cohort/method assignment in {fold}")
    if len(pred) != 2 * EXPECTED_N[fold]:
        raise ValueError(f"Incorrect prediction count in {fold}")
    for method in METHODS:
        part = pred[pred["method"] == method]
        if len(part) != EXPECTED_N[fold] or not part["ID"].is_unique:
            raise ValueError(f"Duplicate or missing predictions in {fold}/{method}")
        if set(part["y_true"].unique()) != {0, 1} or not np.isfinite(part["probability_CD"]).all() or not part["probability_CD"].between(0, 1).all():
            raise ValueError(f"Invalid probabilities or labels in {fold}/{method}")
        if (part["y_true"].eq(1) != part["class_true"].eq("CD")).any():
            raise ValueError(f"Inconsistent outcome mapping in {fold}/{method}")
    p0 = pred[pred.method == METHODS[0]].set_index("ID").sort_index()
    p1 = pred[pred.method == METHODS[1]].set_index("ID").sort_index()
    if list(p0.index) != list(p1.index) or not p0["y_true"].equals(p1["y_true"]):
        raise ValueError(f"The two methods did not predict identical samples in {fold}")
    for result in report["results"]:
        method = result["method"]
        if method not in METHODS:
            raise ValueError("Unexpected method in saved run")
        subset = genes[genes.method == method]
        if len(subset) != int(result["nonzero_coefficients"]):
            raise ValueError(f"Selected-gene count mismatch in {fold}/{method}")
        if not subset["gene"].is_unique:
            raise ValueError(f"Duplicate selected genes in {fold}/{method}")
        all_parameters.append({k: result.get(k) for k in ("outer", "method", "C", "l1_ratio", "inner_mean_auroc", "nonzero_coefficients", "candidate_gene_count", "train_patients", "test_patients", "converged", "n_iter")})
    if not inner["outer"].eq(fold).all() or len(inner) != 36:
        raise ValueError(f"Unexpected inner validation scores in {fold}")
    if not inner["converged"].all() or any(not row.get("converged", False) for row in report["results"]):
        convergence_warnings.append(fold)
    software_versions[fold] = report.get("software")
    all_predictions.append(pred)
    all_inner.append(inner)
    all_genes.append(genes)

predictions = pd.concat(all_predictions, ignore_index=True)
inner_scores = pd.concat(all_inner, ignore_index=True)
selected_genes = pd.concat(all_genes, ignore_index=True)
parameters = pd.DataFrame(all_parameters)
if any(predictions[predictions.method == method].ID.nunique() != 689 for method in METHODS):
    raise ValueError("Incomplete patient coverage")

save_csv(predictions, "all_heldout_predictions.csv")
save_csv(parameters, "table_hyperparameters_and_signature_sizes.csv")
save_csv(inner_scores, "table_inner_validation.csv")
save_csv(selected_genes, "table_selected_genes.csv")
source_manifest = {
    fold: {"origin": origins[fold], "report_sha256": sources[fold][2], "report": sources[fold][0]}
    for fold in COHORTS
}
(run_dir / "input_run_reports.json").write_text(json.dumps(source_manifest, indent=2, sort_keys=True), encoding="utf-8")
print("Input integrity, configuration agreement and patient coverage: PASS")

D1: verified runs/full_gene_loco/d1/report.json
D2: verified runs/full_gene_loco/d2/report.json
D3: verified runs/full_gene_loco/d3/report.json
D4: verified runs/full_gene_loco/d4/report.json
Input integrity, configuration agreement and patient coverage: PASS


In [3]:
def metric_record(subset):
    y = subset["y_true"].to_numpy(dtype=int)
    p = subset["probability_CD"].to_numpy(dtype=float)
    pred = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "n": len(y), "cases": int(sum(y)), "controls": int(len(y) - sum(y)),
        "auroc": float(roc_auc_score(y,p)),
        "average_precision": float(average_precision_score(y,p)),
        "balanced_accuracy": float(balanced_accuracy_score(y,pred)),
        "f1": float(f1_score(y,pred,zero_division=0)),
        "precision": float(precision_score(y,pred,zero_division=0)),
        "sensitivity": float(recall_score(y,pred,zero_division=0)),
        "specificity": float(tn/(tn+fp)),
        "mcc": float(matthews_corrcoef(y,pred)),
        "brier_score": float(brier_score_loss(y,p)),
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)
    }

def stratified_bootstrap_auc(subset):
    y = subset["y_true"].to_numpy(dtype=int)
    p = subset["probability_CD"].to_numpy(dtype=float)
    pos = np.flatnonzero(y==1)
    neg = np.flatnonzero(y==0)
    values = np.empty(BOOTSTRAPS)
    for k in range(BOOTSTRAPS):
        idx = np.r_[rng.choice(pos,len(pos),replace=True), rng.choice(neg,len(neg),replace=True)]
        values[k] = roc_auc_score(y[idx],p[idx])
    return tuple(np.quantile(values,[0.025,0.975]))

rows=[]
for fold in COHORTS:
    for method in METHODS:
        subset=predictions[(predictions.outer==fold)&(predictions.method==method)]
        metrics=metric_record(subset)
        bounds=stratified_bootstrap_auc(subset)
        rows.append({"outer":fold,"method":method,**metrics,"auroc_ci_lower":float(bounds[0]),"auroc_ci_upper":float(bounds[1])})
metrics_df=pd.DataFrame(rows)
save_csv(metrics_df,"table_heldout_performance.csv")

macro=[]
for method in METHODS:
    part=metrics_df[metrics_df.method==method]
    record={"method":method,"n_cohorts":len(part)}
    for name in ("auroc","average_precision","balanced_accuracy","f1","sensitivity","specificity","mcc","brier_score"):
        record[f"{name}_mean"]=float(part[name].mean())
        record[f"{name}_sd"]=float(part[name].std(ddof=1))
    macro.append(record)
macro_df=pd.DataFrame(macro)
save_csv(macro_df,"table_macro_method_summary.csv")

pairing=[]
for fold in COHORTS:
    one=predictions[(predictions.outer==fold)&(predictions.method=="lasso")].set_index("ID").sort_index()
    two=predictions[(predictions.outer==fold)&(predictions.method=="elastic_net")].set_index("ID").sort_index()
    y=one.y_true.to_numpy(dtype=int)
    p1=one.probability_CD.to_numpy(dtype=float)
    p2=two.probability_CD.to_numpy(dtype=float)
    pos=np.flatnonzero(y==1)
    neg=np.flatnonzero(y==0)
    deltas=[]
    for _ in range(BOOTSTRAPS):
        idx=np.r_[rng.choice(pos,len(pos),replace=True),rng.choice(neg,len(neg),replace=True)]
        deltas.append(roc_auc_score(y[idx],p2[idx])-roc_auc_score(y[idx],p1[idx]))
    low,high=np.quantile(deltas,[0.025,0.975])
    pairing.append({"outer":fold,"elastic_net_minus_lasso_auroc":float(roc_auc_score(y,p2)-roc_auc_score(y,p1)),"ci_lower":float(low),"ci_upper":float(high)})
save_csv(pd.DataFrame(pairing),"table_paired_auroc_differences.csv")

sets={(method,cohort):set(selected_genes[(selected_genes.method==method)&(selected_genes.outer==cohort)].gene) for method in METHODS for cohort in COHORTS}
overlap=[]
for method in METHODS:
    for i,one in enumerate(COHORTS):
        for two in COHORTS[i+1:]:
            a,b=sets[(method,one)],sets[(method,two)]
            overlap.append({"method":method,"cohort_a":one,"cohort_b":two,"size_a":len(a),"size_b":len(b),"intersection":len(a&b),"union":len(a|b),"jaccard":len(a&b)/len(a|b) if a|b else np.nan,"interpretation":"exploratory_overlap_training_sets_shared"})
save_csv(pd.DataFrame(overlap),"table_gene_set_overlap.csv")

frequency=[]
for method in METHODS:
    all_symbols=sorted(set.union(*(sets[(method,c)] for c in COHORTS)))
    for gene in all_symbols:
        frequency.append({"method":method,"gene":gene,"outer_fold_selection_count":sum(gene in sets[(method,c)] for c in COHORTS),"denominator":4,"interpretation":"overlapping_training_sets_not_independent_reproducibility"})
save_csv(pd.DataFrame(frequency,columns=["method","gene","outer_fold_selection_count","denominator","interpretation"]),"table_gene_selection_frequency_exploratory.csv")
print(metrics_df[["outer","method","auroc","auroc_ci_lower","auroc_ci_upper","balanced_accuracy","f1"]].to_string(index=False))


outer      method    auroc  auroc_ci_lower  auroc_ci_upper  balanced_accuracy       f1
   d1       lasso 0.865116        0.807768        0.915356           0.632263 0.904232
   d1 elastic_net 0.868829        0.817057        0.915413           0.699104 0.911565
   d2       lasso 0.709784        0.623545        0.792716           0.664842 0.647619
   d2 elastic_net 0.644113        0.560667        0.741791           0.616418 0.577114
   d3       lasso 0.718487        0.586064        0.837535           0.662465 0.648649
   d3 elastic_net 0.733894        0.606443        0.850140           0.635154 0.650602
   d4       lasso 0.735356        0.653349        0.805787           0.640579 0.468966
   d4 elastic_net 0.801976        0.728564        0.860558           0.597812 0.345865


In [4]:
fig,axes=plt.subplots(2,2,figsize=(10,8))
for ax,fold in zip(axes.flat,COHORTS):
    for method in METHODS:
        part=predictions[(predictions.outer==fold)&(predictions.method==method)]
        x,y,_=roc_curve(part.y_true,part.probability_CD)
        score=roc_auc_score(part.y_true,part.probability_CD)
        ax.plot(x,y,label=f"{method} (AUROC {score:.3f})")
    ax.plot([0,1],[0,1],linestyle="--",linewidth=0.8)
    ax.set_title(f"{fold.upper()} held out")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.legend(fontsize=8)
fig.suptitle("Held-out ROC curves")
fig.tight_layout()
save_plot(fig,"figure_roc_by_cohort")

fig,axes=plt.subplots(2,2,figsize=(10,8))
for ax,fold in zip(axes.flat,COHORTS):
    for method in METHODS:
        part=predictions[(predictions.outer==fold)&(predictions.method==method)]
        prec,rec,_=precision_recall_curve(part.y_true,part.probability_CD)
        ap=average_precision_score(part.y_true,part.probability_CD)
        ax.plot(rec,prec,label=f"{method} (AP {ap:.3f})")
    ax.set_title(f"{fold.upper()} held out")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend(fontsize=8)
fig.suptitle("Held-out precision–recall curves")
fig.tight_layout()
save_plot(fig,"figure_pr_by_cohort")

for key,header in [("auroc","Held-out AUROC"),("balanced_accuracy","Held-out balanced accuracy"),("f1","Held-out F1 score")]:
    fig,ax=plt.subplots(figsize=(8,4.5))
    x=np.arange(4)
    for j,method in enumerate(METHODS):
        vals=[metrics_df.loc[(metrics_df.outer==fold)&(metrics_df.method==method),key].iloc[0] for fold in COHORTS]
        ax.bar(x+(j-0.5)*0.38,vals,width=0.38,label=method)
    ax.set_xticks(x,[c.upper() for c in COHORTS])
    ax.set_ylim(0,1)
    ax.set_ylabel(header)
    ax.set_xlabel("Held-out cohort")
    ax.legend(frameon=False)
    save_plot(fig,"figure_"+key+"_by_cohort")

fig,ax=plt.subplots(figsize=(8,4.5))
x=np.arange(4)
for j,method in enumerate(METHODS):
    vals=[len(sets[(method,fold)]) for fold in COHORTS]
    ax.bar(x+(j-0.5)*0.38,vals,width=0.38,label=method)
ax.set_xticks(x,[c.upper() for c in COHORTS])
ax.set_ylabel("Nonzero-coefficient genes")
ax.legend(frameon=False)
save_plot(fig,"figure_signature_size")

for method in METHODS:
    matrix=np.empty((4,4))
    for i,c1 in enumerate(COHORTS):
        for j,c2 in enumerate(COHORTS):
            a,b=sets[(method,c1)],sets[(method,c2)]
            matrix[i,j]=len(a&b)/len(a|b) if a|b else np.nan
    fig,ax=plt.subplots(figsize=(5.6,4.8))
    im=ax.imshow(matrix,vmin=0,vmax=1)
    ax.set_xticks(range(4),[c.upper() for c in COHORTS])
    ax.set_yticks(range(4),[c.upper() for c in COHORTS])
    for i in range(4):
        for j in range(4):
            ax.text(j,i,f"{matrix[i,j]:.2f}" if np.isfinite(matrix[i,j]) else "NA",ha="center",va="center",fontsize=9)
    fig.colorbar(im,ax=ax,label="Jaccard index")
    ax.set_title(f"{method}: LOCO signature overlap (exploratory)")
    save_plot(fig,f"figure_jaccard_{method}")

( run_dir / "METHODS_AND_LIMITATIONS.txt" ).write_text(
    "All reported predictions come from held-out cohorts. Metrics use CD as positive class and 0.5 as a fixed threshold. "
    "AUROC confidence intervals and paired AUROC differences use 1000 stratified bootstrap resamples (seed 355). "
    "Macro summaries use unweighted means and sample standard deviations across four cohorts. "
    "LOCO gene sets are trained on overlapping cohorts, so Jaccard values and selection frequency are exploratory and NOT independent biomarker replication. "
    "Selection frequency does not represent repeated stability-selection resampling. "
    "No pathway enrichment or rank-correlation conclusions are supported by these files. "
    "RNA-seq normalisation is the documented log2(1 + CPM) pilot protocol, not TENTACLES preprocessing.\n",
    encoding="utf-8"
)

manifest={
    "status":"PASS" if not convergence_warnings else "WARN_CONVERGENCE",
    "dataset_sha256": EXPECTED_DATASET,
    "input_reports":{fold:{"origin":origins[fold],"report_sha256":sources[fold][2],"status":sources[fold][0]["status"],"software":software_versions[fold]} for fold in COHORTS},
    "n_patients":689, "methods":list(METHODS),"n_heldout_cohorts":4,
    "bootstrap":{"resamples":BOOTSTRAPS,"seed":SEED,"type":"stratified_case_control"},
    "warnings":{"convergence_folds":convergence_warnings,"gene_overlap":"LOCO training sets overlap; do not interpret as independent cohort replication"},
    "created_utc":datetime.now(timezone.utc).isoformat()
}
manifest["output_sha256"]={p.name:digest(p) for p in sorted(run_dir.iterdir()) if p.is_file()}
(run_dir / "report.json").write_text(json.dumps(manifest,indent=2,sort_keys=True),encoding="utf-8")
print("REPORT_STATUS=",manifest["status"])
print("Number of output files:",len(manifest["output_sha256"])+1)
print("Directory:",run_dir)


REPORT_STATUS= PASS
Number of output files: 28
Directory: /content/drive/MyDrive/CMPS355_Research/runs/full_gene_results_report/20261107T091750523128Z
